#### Final Presentation Schedule

<strong>Pipeline</strong>
- Normalize form data.
- Select candidate slots for `day1`.
- Convert people to project groups.
- Compute group-level availability intersections.
- Assign groups to `day1` first, then solve `day2` from leftovers.
- Generate presentation times.
- Randomize within each day until edge-slot constraints are satisfied.
- Patch remaining outliers at the edges if needed (and availability overlaps).

In [ ]:
import hmac
import hashlib
import unicodedata
import pandas as pd

SECRET_KEY = b"lmao"

def pseudonymize_name(name: str, prefix: str="Student") -> str:
    """Deterministically pseudonymize Chinese/English names."""
    if pd.isna(name):
        return name

    normalized = unicodedata.normalize("NFKC", str(name)).strip()

    digest = hmac.new(SECRET_KEY,
                      normalized.encode("utf-8"),
                      hashlib.sha256).hexdigest()
    return f"{prefix}_{digest[:8]}"

In [1]:
import pandas as pd
from pandas.api.types import is_string_dtype
import numpy as np
from datetime import datetime, timedelta
import random
from dataclasses import dataclass

In [ ]:
DIRECTORY = "some_path"
FILE_NAME = "MasterSheet.xlsx"
SHEET_NAME = "StudentsLog"
AVAILABILITY_FILE = "Final Presentation Availability (Responses).xlsx"

NCU_FINAL_INT = 16
FINAL_LENGTH = 10.0       # minutes
FINAL_QA_LENGTH = 2.0     # minutes
MAX_NUM_PER_SESSION = 10
INSTRUCTORS_AND_TAS = ["Josh", "Charlene", 
                       "Annabel", "Darren", "Lizhu",
                       "Kevin", "Tzu-Yu",
                       "Amanda", "YongSheng", "Trevor", "Tzu-Yun", 
                       "Johanna", "Xiaoan", "Jia-Li",
                       "Ting"]
FINAL_DATES = {"fri": (2026, 6, 12),
               "mon": (2026, 6, 15)}

SEED = 42

In [7]:
master = pd.read_excel(f"{DIRECTORY}/{FILE_NAME}", sheet_name=SHEET_NAME)
master["-> SG"] = master["-> SG"].astype(str)
master = master.loc[(~ master["-> SG"].str.contains("withdrew")) & (~ master["-> SG"].str.contains("audit"))]
master = master.rename(mapper=lambda s: s.strip().lower(), axis=1)
master = master[["name",
         "id",
         "school",
         "dept",
         "course",
         "discord",
         "github",
         "team mates",
         "constraint"]]
master["name"] = master["name"].map(lambda s: s.strip())
master["school"] = master["school"].map(lambda s: s.strip())
master = master.loc[(~master["school"].str.contains("NCU"))
            & (~master["team mates"].str.contains("withdrew"))
            & (~master["team mates"].str.contains("auditor"))]
master["team mates"] = master["team mates"].map(lambda s: "" if isinstance(s, float) else s)
master["team mates"] = master["team mates"].map(lambda s: s.split(", "))
master["team mates"] = master["team mates"].map(lambda lst: [s.strip() for s in lst])
master = master.reset_index(drop=True)

In [18]:
def mapper(s):
    if "Fri" in s:
        return "fri"
    elif "Mon" in s:
        return "mon"
    else:
        return s.lower()
df =  pd.read_excel(f"{DIRECTORY}/{AVAILABILITY_FILE}")
df = df.rename(mapper=mapper, axis=1)
#df = df.loc[~ df["name"].isin(INSTRUCTORS_AND_TAS)]

def get_slots(df: pd.DataFrame, day: str, use_all_available: bool=False) -> list:
    # this will modify the input df
    if is_string_dtype(df[day]):
        df[day] = df[day].map(lambda s: s.split(", ")).map(lambda lst: [int(s[:2]) if not s.startswith("N")
                                                                                   else -1 for s in lst])
    else:
        assert df[day].copy().map(lambda s: isinstance(s, set)).all()
        df[day] = df[day].map(lambda s: list(s))

    day_col = df[day].copy()
    u, c = np.unique(np.array(sum(day_col, [])), return_counts=True)
    if use_all_available:
        u_list = list(u)
        return u_list
    counts = dict(zip(u, c))
    opt = u[np.argmax(c)]
    assert opt != -1, f"For {day}, mode (most frequent) is 'None of the above'..."

    if counts.get(opt + 1, -1) >= counts.get(opt - 1, -1):
        sec = opt + 1
    else:
        sec = opt - 1
    return [opt, sec]


def get_people(df: pd.DataFrame, **kwargs) -> tuple:
    def mask(lst: list, *, s: list):
        if set(lst) & set(s)  != set():
            return True
        else:
            return False
    def check_available(lst: list, *, s: list):
        return set(lst) & set(s)
    
    if "name" in df.columns:
        df = df.rename({"name": "group"}, axis=1)

    days_slots, days_df = {}, {}
    for day, boo in kwargs.items():
        slots = get_slots(df, day, use_all_available=boo)
        if day == "mon" and NCU_FINAL_INT not in slots:
            slots.append(NCU_FINAL_INT)

        days_slots[day] = slots
        days_df[day] = df.loc[df[day].copy().map(mask,
                                                 s=days_slots[day])]
    _df = df.copy()
    _df = _df.rename({"group": "name"}, axis=1)

    not_assigned = set(df["group"])
    for day, ddf in days_df.items():
        ddf[day] = ddf[day].map(check_available,
                                s=days_slots[day])
        if "timestamp" in ddf.columns:
            ddf.drop(columns=["timestamp"] + [d for d in days_df.keys() if d != day], inplace=True)
        not_assigned -= (set(ddf["group"]) & not_assigned)
    
    return days_slots, days_df, not_assigned, _df  # (dict of lists, dict of data frames, set)


def get_groups(master):
    groups = [tuple(sorted([master.iloc[i]["name"]] + master.iloc[i]["team mates"]) + [master.iloc[i]["school"]])
                                                  for i in range(master.shape[0])]
    groups = set(groups)
    groups = [tuple([s for s in group if s!= ""]) for group in groups]
    return groups


def convert_to_group(days_df: dict, groups: list) -> dict: # returns dict of data frames
    def helper(name: str, *, groups: list):
        for group in groups:
            if name in group:
                return group
        # instructors and TAs aren't in groups
        return name 
    for ddf in days_df.values():
        if "group" not in ddf.columns:
            raise ValueError(f"""Please rename the 'name' column to 'group'
        and ensure that it's already grouped (type should be tuple, not str) before passing it to this function""")
        ddf["group"] = ddf["group"].map(helper, groups=groups)
    return days_df


def get_group_time_intersection(days_df: dict) -> dict: # returns dict of data frames
    def per_ddf(day: str, ddf: pd.DataFrame):
        if "group" not in ddf.columns:
            raise ValueError(f"""For the {day} availability sub-dataframe, please rename the 'name' column to 'group'
        and ensure that it's already grouped (type should be tuple, not str) before passing it to this function""")

        groups_col = ddf["group"]
        to_get_intersect = [g for g, v in dict(groups_col.value_counts()).items() if v > 1]

        to_concat = pd.DataFrame()
        for g in to_get_intersect:
            rows = ddf.loc[ddf["group"] == g].copy()
            slots = rows[day].to_list()
            intersect = slots[0]
            for s in slots[1:]:
                intersect = intersect & s
            to_concat = pd.concat([to_concat,
                                   pd.DataFrame({"group": [g], day: [intersect]})])
        
        return pd.concat([ddf.loc[~ ddf["group"].isin(to_get_intersect)],
                          to_concat])
        
    return_days_df = {}
    for day, ddf in days_df.items():
        return_days_df[day] = per_ddf(day, ddf)
    return return_days_df


def generate_times(days_slots: dict) -> dict:
    def per_day(day: str, slots: list):
        def helper(hr: int):
            yyyy, mm, dd = FINAL_DATES[day]
            start = datetime(yyyy, mm, dd, hr, 00)
            times = [start + timedelta(minutes=(FINAL_LENGTH + FINAL_QA_LENGTH) * i)
                            for i in range(MAX_NUM_PER_SESSION)]
            return times
        
        if len(slots) == 2:
            times = helper(min(slots))
        
        elif len(slots) >= 3:
            if np.array_equal(np.diff(np.sort(slots)), np.array([1, 1])):
                times = helper(min(slots))
            else:
                _slots = slots.copy()
                _slots.remove(NCU_FINAL_INT)
                times1, times2 = helper(min(_slots)), helper(NCU_FINAL_INT)
                times = (times1, times2)
        return times
    
    days_times = {}
    for day, slots in days_slots.items():
        days_times[day] = per_day(day, slots)
    return days_times


def assign_to_date(days_df: dict, days_slots: dict, *, day1: str, day2: str) -> dict:
    """Assign to `day1` first."""
    df1, df2 = days_df[day1].copy(), days_df[day2].copy()
    df1 = df1.loc[~ df1["group"].isin(INSTRUCTORS_AND_TAS)]
    df2 = df2.loc[~ df2["group"].isin(INSTRUCTORS_AND_TAS)]
    groups2 = set(df2["group"])

    slots = list(days_slots[day1].copy())
    if NCU_FINAL_INT in slots:
        slots.remove(NCU_FINAL_INT)

    #mask1 = [s & set(slots) == set(slots) for s in df1[day1].to_list()]
    #df1.insert(0, "mask1", mask1)
    mask2 = df2.loc[df2[day2] == set([-1])]["group"]
    #priority = df1.loc[df1["mask1"] | (df1["group"].isin(mask2))]
    priority = df1.loc[df1["group"].isin(mask2)]
    num = len(priority)
    groups1 = set(priority["group"])
    assert len(groups1) == num

    if num <= MAX_NUM_PER_SESSION:
        remain = list(set(df1["group"]) - groups1)
        random.shuffle(remain)
        for group in remain:
            if num >= MAX_NUM_PER_SESSION:
                break
            if df1.loc[df1["group"] == group].iloc[0][day1] != set(): # prevents empty set after get_group_intersection
                groups1 = groups1 | set([group])
                num += 1
    else:
        overflow = list(groups1)
        random.shuffle(overflow)
        for group in overflow:
            if num == MAX_NUM_PER_SESSION:
                break
            groups1 -= set([group])
            num -= 1

    groups2 -= groups2 & groups1
    #df1 = df1.drop(columns="mask1")
    df1 = df1.loc[df1["group"].isin(groups1)]
    df2 = df2.loc[df2["group"].isin(groups2)]
    return {day1: df1, day2: df2}


def check_constraints(day: str, days_slots: dict, days_times: dict, ddf_sampled: pd.DataFrame) -> bool:
    if not ddf_sampled.index.equals(pd.RangeIndex(len(ddf_sampled))):
        raise ValueError("ddf_sampled must be reindexed.")

    slots, times = days_slots[day], days_times[day]
    _slots = slots.copy()
    _ddf_sampled = ddf_sampled.copy()

    if len(_slots) == 3 and not np.array_equal(np.diff(np.sort(_slots)), np.array([1, 1])):
        def helper(s: set):
            s_list = list(s)
            if NCU_FINAL_INT in s_list:
                s_list.remove(NCU_FINAL_INT)
            return set(s_list)

        slots_col = ddf_sampled[day].to_list()
        if set([NCU_FINAL_INT]) in slots_col:
            raise ValueError (f"""Please filter out groups whose only availability for {day} is {NCU_FINAL_INT}: 00
            before randomizing in reindexing the data frame.""")

        _slots.remove(NCU_FINAL_INT)
        _ddf_sampled[day] = _ddf_sampled[day].map(helper)
        assert isinstance(times, tuple) and len(times) == 2
        times = times[0]

    yyyy, mm, dd = FINAL_DATES[day]

    def check_func(s: set, t: datetime):
        # s is the cell value, t is assigned time
        try:
            if list(s)[0] == min(_slots):
                return True if t < datetime(yyyy, mm, dd, min(_slots)+1, 00) - timedelta(minutes=(FINAL_LENGTH + FINAL_QA_LENGTH) * 2) else False   
            elif list(s)[0] == max(_slots):
                return True if t > datetime(yyyy, mm, dd, max(_slots), 00) else False
        except IndexError:
            print("IndexError! Something wrong with code -- fix it!")
            pass
    
    has_constraint = _ddf_sampled.loc[_ddf_sampled[day] != set(_slots)].copy()
    check = [check_func(s, times[idx]) for s, idx in zip(has_constraint[day], has_constraint.index)]

    return all(check)

@dataclass
class Results:
    schedules: dict[str, pd.DataFrame]
    merge_with_ncu: pd.DataFrame
    not_assigned: set
    df_formatted: pd.DataFrame
    num_iter: int

    def format(self):
        def helper(df):
            def extract_school(cell, get="name"):
                if isinstance(cell, tuple):
                    if get == "name":
                        return cell[:-1]
                    elif get == "school":
                        return cell[-1]
                else:
                    return cell
            s1 = df["group"].copy().map(extract_school, get="name")
            s2 = df["group"].copy().map(extract_school, get="school")
            if "time" in df.columns:
                data = {"TIME": df["time"],
                        "GROUP": s1,
                        "INSTITUTE": s2}
            else:
                data = {"GROUP": s1,
                        "INSTITUTE": s2}
            _df = pd.DataFrame(data)
            return _df
        
        def clean(cell):
            if isinstance(cell, tuple):
                if len(cell) == 1:
                    return cell[0]
                else:
                    return " + ".join(cell)
            else:
                return cell
            
        schedules = self.schedules.copy()
        for day, schedule in self.schedules.items():
            schedules[day] = helper(schedule).map(clean)
        self.schedules = schedules
        self.merge_with_ncu = helper(self.merge_with_ncu).map(clean)
    
    def get_students_no_response(self, master: pd.DataFrame):
        if "name" not in master.columns:
            raise ValueError("'name' should be in master sheet columns")
        all_names = master["name"]
        assert len(all_names) == len(set(all_names))
        all_names = set(all_names)
        names_responded = set(self.df_formatted["name"])
        return all_names - (all_names & names_responded)
    
    def save(self, out_path: str, index: bool=True):
        sheet_name_dict = {"mon": "Monday (June 15)",
                           "fri": "Friday (June 12)"}
        with pd.ExcelWriter(f"{out_path}.xlsx") as writer:
            for day, schedule in self.schedules.items():
                schedule.to_excel(writer, sheet_name=sheet_name_dict[day], index=index)
            self.merge_with_ncu.to_excel(writer, sheet_name="Merge with NCU (June 15, 4 pm)", index=index)


def get_schedule(df: pd.DataFrame, master: pd.DataFrame, *, day1: str, day2: str, verbose: bool=False) -> tuple:
    kwargs = {day1: False, day2: True}
    days_slots, days_df, _, _df = get_people(df, **kwargs)
    groups = get_groups(master)
    days_df = get_group_time_intersection(convert_to_group(days_df, groups))

    def solve(days_df: dict, j: int) -> Results | None:
        if verbose:
            print(f">> OUTER LOOP: ITERATION {j+1}")
        output = assign_to_date(days_df, days_slots, day1=day1, day2=day2)
        df1 = output[day1]
        slots1 = days_slots[day1]
        
        kwargs = {day2: False}
        slots2_dict, df2_dict, not_assigned, _ = get_people(output[day2], **kwargs)
        if len(not_assigned) > 3:
            if verbose:
                print(not_assigned)
            return None
        
        df2 = df2_dict[day2]
        #df2 = df2.loc[~ df2["group"].isin(INSTRUCTORS_AND_TAS)]
        new_days_df = {day1: df1, day2: df2}
        new_days_slots = {day1: slots1, day2: slots2_dict[day2]}

        days_times = generate_times(new_days_slots)
        ddf_for_schedule = {}
        to_merge_with_ncu = pd.DataFrame()
        for day, ddf in new_days_df.items():
            if day == "mon":
                to_merge_with_ncu = ddf.loc[ddf[day] == set([NCU_FINAL_INT])]
                ddf = ddf.loc[ddf[day] != set([NCU_FINAL_INT])]
            
            check = False
            i = 0
            while check is False:
                i += 1
                if i > 30:
                    if verbose:
                        print(f"{day}: failed to converge...")
                        display(ddf)
                    return None
                if len(to_merge_with_ncu) > 2:
                    continue
                ddf = ddf.sample(frac=1)
                ddf = ddf.reset_index(drop=True)
                check = check_constraints(day, new_days_slots, days_times, ddf)
            times = days_times[day]
            if isinstance(times, tuple):
                times = times[0]
            ts = pd.DataFrame({"time": times})
            ts = ts.map(lambda t: t.strftime("%H:%M:%S")).iloc[:len(ddf)]
            ddf = pd.concat([ts, ddf], axis=1)
            ddf_for_schedule[day] = ddf
        
        results = Results(ddf_for_schedule, to_merge_with_ncu, not_assigned, _df, i)
        return results
    
    status = None
    j = 0
    while status is None:
        status = solve(days_df, j)
        j += 1
        if j > 30:
            print(f"Max outer loop iter {j} hit! Constraint satisfaction problem failed to converge LOL...")
            break

    return status, j 


def shift_schedule(day: str, schedule: pd.DataFrame, buffer: int) -> pd.DataFrame:
    if day != "mon":
        print("This function is intended for shifting the Monday schedule so it's practical and aligns with NCU's scheduled session.")
        return schedule

    yyyy, mm, dd = FINAL_DATES[day][0], FINAL_DATES[day][1], FINAL_DATES[day][2]
    last_time_str = schedule.iloc[-1]["time"]
    h, m, _ = tuple(last_time_str.split(":"))
    last_time = datetime(yyyy, mm, dd, int(h), int(m), 00)
    delta = datetime(yyyy, mm, dd, NCU_FINAL_INT, 00, 00) - last_time
    float_delta = delta.total_seconds() / 60.0
    
    if abs(float_delta) > 15.0:
        def shift(time_str):
            h, m, _ = tuple(time_str.split(":"))
            t = datetime(yyyy, mm, dd, int(h), int(m), 00) + delta - timedelta(minutes=(FINAL_LENGTH+FINAL_QA_LENGTH) * (1-buffer))
            return t.strftime("%H:%M:%S")
        schedule["time"] = schedule["time"].map(shift)
        return schedule

    elif 0 < float_delta < 15.0:
        print(f"There's only {float_delta} minutes to NCU's session... Just take a short break LMAO.")
        return schedule
    
    else:
        print(f"Push NCU's session later by {float_delta}... I guess that's okay?")
        return schedule


def manual_patch(status: Results):
    df_formatted = status.df_formatted
    schedules = status.schedules
    not_assigned = status.not_assigned

    def check(day: str, schedule: pd.DataFrame, group: tuple):
        def to_interval(time_str: str, delta: float, subtract: bool) -> tuple: # delta in minutes
            h, m, _ = tuple(time_str.split(":"))
            t1 = datetime(2026, 1, 1, int(h), int(m), 00) # only want the hours and minutes; yyyy, mm, dd are dummies
            assert delta > 0
            if subtract:
                t2 = t1 - timedelta(minutes=delta)
                return (t2.hour + t2.minute/60.0, t1.hour + t1.minute/60.0)
            else:
                t2 = t1 + timedelta(minutes=delta)
                return (t1.hour + t1.minute/60.0, t2.hour + t2.minute/60.0)
            
        def overlap_or_not(tup1: tuple, tup2: tuple) -> bool:
            min1, min2 = min(tup1), min(tup2)
            max1, max2 = max(tup1), max(tup2)
            if min1 < max2 and min2 < max1:
                return True
            else:
                return False
            
        rows = df_formatted.loc[df_formatted["name"].isin(group)].copy()
        avails = rows[day].to_list()
        if len(rows) > 1:
            intersect = set(sum(avails, []))
            for a in avails:
                intersect = intersect & set(a)
            _avail = list(intersect)
        else:
            _avail = avails[0]
        if _avail and -1 not in _avail:
            avail = [(a, a+1) for a in _avail]
        else:
            avail = []
        
        start, end = schedule["time"].iloc[0], schedule["time"].iloc[-1]
        interval1 = to_interval(start, 
                                delta=FINAL_LENGTH+FINAL_QA_LENGTH,
                                subtract=True)
        interval2 = to_interval(end,
                                delta=FINAL_LENGTH+FINAL_QA_LENGTH,
                                subtract=False)

        overlap1 = any([overlap_or_not(interval1, a) for a in avail])
        overlap2 = any([overlap_or_not(interval2, a) for a in avail])
        return [overlap1, overlap2]
    
    def to_time(day, time_str: str, delta: float, subtract: bool):
        h, m, _ = tuple(time_str.split(":"))
        t = datetime(FINAL_DATES[day][0], FINAL_DATES[day][1], FINAL_DATES[day][2],
                     int(h), int(m), 00)
        if subtract:
            t -= timedelta(minutes=delta)
        else:
            t += timedelta(minutes=delta)
        return t.strftime("%H:%M:%S")
    
    days = list(schedules.keys())
    _not_assigned = not_assigned.copy()
    for group in not_assigned:
        overlaps = []
        for day, schedule in schedules.items():
            overlap = check(day, schedule, group)
            overlaps.append(overlap)

        overlaps = np.array(overlaps)
        where = np.array(np.where(overlaps)).T
        if where.size > 0:
            idx = where[np.random.randint(len(where))]
            assert overlaps[idx[0], idx[1]]
            day = days[idx[0]]
            schedule = schedules[day]

            if idx[1] == 1: # end
                time = to_time(day,
                               schedule.iloc[-1]["time"],
                               delta=FINAL_LENGTH+FINAL_QA_LENGTH,
                               subtract=False)
                to_concat = pd.DataFrame({"time": [time],
                                      "group": [group],
                                      day: ["*"]})
                schedule = pd.concat([schedule, to_concat])
                
            elif idx[1] == 0: # start
                time = to_time(day,
                               schedule.iloc[0]["time"],
                               delta=FINAL_LENGTH+FINAL_QA_LENGTH,
                               subtract=True)
                to_concat = pd.DataFrame({"time": [time],
                                          "group": [group],
                                          day: ["*"]})
                schedule = pd.concat([to_concat, schedule])
            schedule.reset_index(drop=True, inplace=True)
            schedules[day] = schedule
            _not_assigned.remove(group)
    status.not_assigned = _not_assigned



orders = [dict(day1="fri", day2="mon"),
          dict(day1="mon", day2="fri")]
results_list = []
for order in orders:
    print(f"**Prioritize day1 = {order['day1']}**\n".upper())
    status, j = get_schedule(df, master, **order, verbose=False)
    if isinstance(status, Results):
        print(f"Number of iterations: outer loop = {j}, inner loop = {status.num_iter}")
        print(">> Outliers, not assigned:")
        #print(status.not_assigned)
        print(f"len(not_assigned) = {len(status.not_assigned)}")
        schedule_fri = status.schedules["fri"].copy()
        schedule_mon = status.schedules["mon"].copy()
        schedule_fri["group"] = schedule_fri["group"].map(pseudonymize_name)
        schedule_mon["group"] = schedule_mon["group"].map(pseudonymize_name)
        display(schedule_fri)
        display(schedule_mon)
        print(">> Merge with NCU:")
        merge_with_ncu = status.merge_with_ncu.copy()
        merge_with_ncu["group"] = merge_with_ncu["group"].map(pseudonymize_name)
        display(merge_with_ncu)
    results_list.append(status)
    print("\n")


**PRIORITIZE DAY1 = FRI**

Number of iterations: outer loop = 3, inner loop = 1
>> Outliers, not assigned:
len(not_assigned) = 3


,time,group,fri
0,14:00:00,Student_6751e5fe,{14}
1,14:12:00,Student_9850c5e6,"{14, 15}"
2,14:24:00,Student_10b68ef6,"{14, 15}"
3,14:36:00,Student_98ae1007,"{14, 15}"
4,14:48:00,Student_3ddb6a2c,"{14, 15}"
5,15:00:00,Student_3a9c894d,"{14, 15}"
6,15:12:00,Student_fb311177,"{14, 15}"
7,15:24:00,Student_ea01b8d6,"{14, 15}"
8,15:36:00,Student_47ab4f40,"{14, 15}"
9,15:48:00,Student_6018d999,{15}


,time,group,mon
0,15:00:00,Student_bf817c8b,"{16, 15}"
1,15:12:00,Student_7f05c7c5,"{16, 15}"
2,15:24:00,Student_7b0b0283,"{16, 15}"
3,15:36:00,Student_54e46699,"{16, 15}"
4,15:48:00,Student_415145aa,"{16, 15}"


>> Merge with NCU:


,group,mon
13,Student_a0f7d958,{16}




**PRIORITIZE DAY1 = MON**

Number of iterations: outer loop = 5, inner loop = 3
>> Outliers, not assigned:
len(not_assigned) = 2


,time,group,fri
0,14:00:00,Student_b8fd73ee,{14}
1,14:12:00,Student_10b68ef6,"{14, 15}"
2,14:24:00,Student_6751e5fe,{14}
3,14:36:00,Student_98ae1007,"{14, 15}"
4,14:48:00,Student_fb311177,"{14, 15}"
5,15:00:00,Student_ea01b8d6,"{14, 15}"
6,15:12:00,Student_54e46699,"{14, 15}"


,time,group,mon
0,15:00:00,Student_7f05c7c5,"{16, 15}"
1,15:12:00,Student_9850c5e6,"{16, 15}"
2,15:24:00,Student_7b0b0283,"{16, 15}"
3,15:36:00,Student_415145aa,"{16, 15}"
4,15:48:00,Student_3ddb6a2c,"{16, 15}"
5,16:00:00,Student_bf817c8b,"{16, 15}"
6,16:12:00,Student_47ab4f40,"{16, 15}"
7,16:24:00,Student_6018d999,"{16, 15}"
8,16:36:00,Student_3a9c894d,"{16, 15}"


>> Merge with NCU:


,group,mon
13,Student_a0f7d958,{16}


In [19]:
status = results_list[0]
schedules = status.schedules
buffer = len(status.merge_with_ncu)
schedules["mon"] = shift_schedule(day="mon", schedule=schedules["mon"], buffer=buffer)
status.schedules = schedules

manual_patch(status)
status.format()
no_response = status.get_students_no_response(master)
print("Friday, June 12")
schedule_fri = status.schedules["fri"].copy()
schedule_fri["GROUP"] = schedule_fri["GROUP"].map(pseudonymize_name)
display(schedule_fri)
print("Monday, June 15")
schedule_mon = status.schedules["mon"].copy()
schedule_mon["GROUP"] = schedule_mon["GROUP"].map(pseudonymize_name)
display(schedule_mon)
print("Merge with NCU:")
merge_with_ncu = status.merge_with_ncu.copy()
merge_with_ncu["GROUP"] = merge_with_ncu["GROUP"].map(pseudonymize_name)
display(merge_with_ncu)
print(">> Outliers, not assigned:")
#print(status.not_assigned)
print(f"len(not_assigned) = {len(status.not_assigned)}")
print("\n>> Didn't submit Google Form:")
#print(no_response)
print(f"len(no_response) = {len(no_response)}")

There's only 12.0 minutes to NCU's session... Just take a short break LMAO.
Friday, June 12


,TIME,GROUP,INSTITUTE
0,13:36:00,Student_476807c3,NTU-TW
1,13:48:00,Student_e2fd1d0a,NTU-SG
2,14:00:00,Student_1ecaaa98,NTU-TW
3,14:12:00,Student_7ffbc691,NTU-TW
4,14:24:00,Student_70e9cf76,NTU-SG
5,14:36:00,Student_d2849424,NTU-TW
6,14:48:00,Student_8da30f76,NTU-SG
7,15:00:00,Student_992de68d,NTU-TW
8,15:12:00,Student_3dfe6948,NTU-TW
9,15:24:00,Student_ba6a4dc5,NTU-TW


Monday, June 15


,TIME,GROUP,INSTITUTE
0,15:00:00,Student_1919f482,NTU-SG
1,15:12:00,Student_de2351dd,NTU-TW
2,15:24:00,Student_eaabaae1,NTU-TW
3,15:36:00,Student_279bc611,NTU-TW
4,15:48:00,Student_327cca2c,NTU-TW


Merge with NCU:


,GROUP,INSTITUTE
13,Student_07c2f47b,NTU-TW


>> Outliers, not assigned:
len(not_assigned) = 1

>> Didn't submit Google Form:
len(no_response) = 1
